# Notebook 3: Shadow Credit Rating
## Indian MSME Credit Risk Framework

**Objective:** Derive a shadow credit rating for a fictional Indian NBFC
(Pradhan MSME Lending Institution) using a weighted scorecard across five
dimensions, anchored in stress test results from Notebook 2.

**Rating Framework:**
| Dimension | Weight |
|-----------|--------|
| Asset Quality | 30% |
| Stress Resilience | 25% |
| Portfolio Concentration | 15% |
| GST Compliance | 15% |
| Risk Distribution | 15% |

**Final Rating: BB — Moderate Risk**
**Weighted Score: 2.75 / 5.00**

**Key Finding:** Rating constrained by elevated base default rate (22.12%)
and limited stress resilience (severe ECL rate 21.44%), partially offset
by adequate sectoral diversification and risk band distribution.

In [7]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

# Load stress results and enriched portfolio
stress_df = pd.read_csv('/content/drive/MyDrive/Credit_Risk_Framework/Outputs/stress_results.csv')
portfolio_df = pd.read_csv('/content/drive/MyDrive/Credit_Risk_Framework/Outputs/enriched_portfolio.csv')

print("Stress Results:")
print(stress_df.to_string(index=False))
print("\nPortfolio shape:", portfolio_df.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Stress Results:
       Scenario   Risk Band  Borrowers  Stressed PD  Total EAD (TWD)  Total ECL (TWD)  ECL Rate (%)
           Base    Low Risk       1839         7.88        489137680       17355167.0          3.55
           Base Medium Risk       2541        16.49        338420000       25111847.0          7.42
           Base   High Risk       1620        47.10        180220000       38196656.0         21.19
Moderate Stress    Low Risk       1839        11.04        489137680       24297234.0          4.97
Moderate Stress Medium Risk       2541        23.09        338420000       35156586.0         10.39
Moderate Stress   High Risk       1620        65.94        180220000       53475318.0         29.67
  Severe Stress    Low Risk       1839        15.77        489137680       34710335.0          7.10
  Severe Stress Medium Risk       2541        32.98    

In [8]:
# Shadow Rating Framework
# We score the institution across 5 dimensions

# 1. Asset Quality Score (based on base default rate)
base_default_rate = portfolio_df['DEFAULT'].mean()

def asset_quality_score(dr):
    if dr < 0.10: return 5
    elif dr < 0.15: return 4
    elif dr < 0.20: return 3
    elif dr < 0.25: return 2
    else: return 1

# 2. Portfolio Concentration Score (sector concentration)
sector_concentration = portfolio_df['SECTOR'].value_counts(normalize=True).max()

def concentration_score(conc):
    if conc < 0.25: return 5
    elif conc < 0.30: return 4
    elif conc < 0.35: return 3
    elif conc < 0.40: return 2
    else: return 1

# 3. Stress Resilience Score (based on severe stress ECL rate)
severe_ecl_rate = stress_df[stress_df['Scenario'] == 'Severe Stress']['ECL Rate (%)'].mean() / 100

def stress_resilience_score(ecl):
    if ecl < 0.10: return 5
    elif ecl < 0.15: return 4
    elif ecl < 0.20: return 3
    elif ecl < 0.25: return 2
    else: return 1

# 4. GST Compliance Score
gst_compliance = portfolio_df['GST_COMPLIANT'].mean()

def gst_score(gst):
    if gst > 0.85: return 5
    elif gst > 0.75: return 4
    elif gst > 0.65: return 3
    elif gst > 0.55: return 2
    else: return 1

# 5. Risk Distribution Score (proportion of low risk borrowers)
low_risk_pct = (portfolio_df['RISK_BAND'] == 'Low Risk').mean()

def risk_distribution_score(lr):
    if lr > 0.40: return 5
    elif lr > 0.30: return 4
    elif lr > 0.25: return 3
    elif lr > 0.20: return 2
    else: return 1

# Calculate scores
scores = {
    'Asset Quality':        asset_quality_score(base_default_rate),
    'Portfolio Concentration': concentration_score(sector_concentration),
    'Stress Resilience':    stress_resilience_score(severe_ecl_rate),
    'GST Compliance':       gst_score(gst_compliance),
    'Risk Distribution':    risk_distribution_score(low_risk_pct)
}

print("Dimension Scores (out of 5):")
for dim, score in scores.items():
    print(f"  {dim}: {score}/5")

print(f"\nRaw inputs:")
print(f"  Base default rate: {base_default_rate:.2%}")
print(f"  Sector concentration: {sector_concentration:.2%}")
print(f"  Severe stress ECL rate: {severe_ecl_rate:.2%}")
print(f"  GST compliance: {gst_compliance:.2%}")
print(f"  Low risk proportion: {low_risk_pct:.2%}")

Dimension Scores (out of 5):
  Asset Quality: 2/5
  Portfolio Concentration: 4/5
  Stress Resilience: 2/5
  GST Compliance: 3/5
  Risk Distribution: 4/5

Raw inputs:
  Base default rate: 22.12%
  Sector concentration: 29.47%
  Severe stress ECL rate: 21.44%
  GST compliance: 68.52%
  Low risk proportion: 30.65%


In [9]:
# Weighted average score
weights = {
    'Asset Quality': 0.30,
    'Portfolio Concentration': 0.15,
    'Stress Resilience': 0.25,
    'GST Compliance': 0.15,
    'Risk Distribution': 0.15
}

weighted_score = sum(scores[dim] * weights[dim] for dim in scores)

print(f"Weighted Score: {weighted_score:.2f} / 5.00")

# Map to rating
def assign_rating(score):
    if score >= 4.5: return 'AAA', 'Highest Safety'
    elif score >= 4.0: return 'AA', 'High Safety'
    elif score >= 3.5: return 'A', 'Adequate Safety'
    elif score >= 3.0: return 'BBB', 'Moderate Safety'
    elif score >= 2.5: return 'BB', 'Moderate Risk'
    elif score >= 2.0: return 'B', 'High Risk'
    else: return 'C', 'Very High Risk'

rating, outlook = assign_rating(weighted_score)

print(f"\nShadow Rating: {rating}")
print(f"Outlook: {outlook}")

# Weighted score breakdown
print("\nScore Breakdown:")
for dim, w in weights.items():
    contribution = scores[dim] * w
    print(f"  {dim} ({w*100:.0f}% weight): {scores[dim]}/5 → contribution {contribution:.2f}")

Weighted Score: 2.75 / 5.00

Shadow Rating: BB
Outlook: Moderate Risk

Score Breakdown:
  Asset Quality (30% weight): 2/5 → contribution 0.60
  Portfolio Concentration (15% weight): 4/5 → contribution 0.60
  Stress Resilience (25% weight): 2/5 → contribution 0.50
  GST Compliance (15% weight): 3/5 → contribution 0.45
  Risk Distribution (15% weight): 4/5 → contribution 0.60


In [10]:
# Generate Shadow Rating Report
from datetime import date

report = f"""
SHADOW CREDIT RATING REPORT
============================
Fictional Entity: Pradhan MSME Lending Institution (PMLI)
Analyst: Priti Dutta
Date: {date.today().strftime("%B %d, %Y")}
Rating: {rating} — {outlook}

1. OVERVIEW
-----------
This report presents a shadow credit rating for Pradhan MSME Lending
Institution (PMLI), a fictional mid-sized NBFC operating across five
Indian regions with a diversified MSME borrower base of 6,000 accounts.
The rating is derived from a quantitative scorecard assessing five
dimensions: asset quality, portfolio concentration, stress resilience,
GST compliance, and risk distribution.

2. RATING RATIONALE
-------------------
PMLI is assigned a shadow rating of {rating} ({outlook}), reflecting a
weighted score of {weighted_score:.2f} out of 5.00. The rating is
constrained primarily by elevated asset quality concerns and limited
stress resilience, partially offset by adequate portfolio diversification
and risk band distribution.

3. DIMENSION ANALYSIS
---------------------
a) Asset Quality [Score: {scores['Asset Quality']}/5 | Weight: 30%]
   Base default rate of {base_default_rate:.2%} places PMLI in the
   moderate-to-high risk category. Approximately 27% of the portfolio
   falls in the High Risk band with a default rate of 47.10%, indicating
   meaningful tail risk concentration.

b) Portfolio Concentration [Score: {scores['Portfolio Concentration']}/5 | Weight: 15%]
   Sector concentration is manageable at {sector_concentration:.2%}, led
   by Retail & Trade (29.47%) and Manufacturing (25.57%). No single sector
   dominates excessively, reflecting reasonable diversification across
   Indian MSME segments.

c) Stress Resilience [Score: {scores['Stress Resilience']}/5 | Weight: 25%]
   Under a severe stress scenario (GDP growth 1.5%, repo rate 9.0%,
   unemployment 12.0%), portfolio ECL rate escalates to {severe_ecl_rate:.2%},
   representing a doubling from the base case of 8.00%. The High Risk
   segment PD approaches 94.20% under severe stress, indicating limited
   shock absorption capacity in the tail.

d) GST Compliance [Score: {scores['GST Compliance']}/5 | Weight: 15%]
   GST compliance stands at {gst_compliance:.2%} across the portfolio.
   While compliance among Low Risk borrowers is strong at 89.5%, the
   High Risk segment shows only 40.1% compliance, suggesting informal
   operations and weaker financial discipline among vulnerable borrowers.

e) Risk Distribution [Score: {scores['Risk Distribution']}/5 | Weight: 15%]
   30.65% of borrowers fall in the Low Risk band, with 42.35% in Medium
   Risk and 27% in High Risk. While the distribution is not adverse,
   a higher Low Risk concentration would be needed to support an
   investment-grade rating.

4. SCENARIO ANALYSIS SUMMARY
-----------------------------
Scenario          | Portfolio ECL Rate | Change vs Base
------------------|--------------------|---------------
Base              |        8.00%       |      —
Moderate Stress   |       11.21%       |    +3.21%
Severe Stress     |       16.01%       |    +8.01%

5. KEY RATING SENSITIVITIES
----------------------------
UPWARD PRESSURES (rating improvement):
- Sustained reduction in High Risk borrower concentration below 20%
- Improvement in GST compliance above 80% portfolio-wide
- Base default rate declining below 15% over two consecutive periods

DOWNWARD PRESSURES (rating deterioration):
- Macro deterioration beyond severe stress assumptions
- Sector concentration exceeding 40% in any single segment
- High Risk PD remaining above 50% under base conditions

6. CONCLUSION
-------------
PMLI's {rating} shadow rating reflects a lending institution with adequate
diversification but meaningful vulnerability to asset quality deterioration
and macro stress. The institution would benefit from tightening its
scorecard cutoffs to reduce High Risk originations and improving GST
compliance monitoring as a leading indicator of borrower financial health.

Rating: {rating} | Outlook: {outlook} | Weighted Score: {weighted_score:.2f}/5.00
"""

print(report)


SHADOW CREDIT RATING REPORT
Fictional Entity: Pradhan MSME Lending Institution (PMLI)
Analyst: Priti Dutta
Date: May 27, 2026
Rating: BB — Moderate Risk

1. OVERVIEW
-----------
This report presents a shadow credit rating for Pradhan MSME Lending 
Institution (PMLI), a fictional mid-sized NBFC operating across five 
Indian regions with a diversified MSME borrower base of 6,000 accounts. 
The rating is derived from a quantitative scorecard assessing five 
dimensions: asset quality, portfolio concentration, stress resilience, 
GST compliance, and risk distribution.

2. RATING RATIONALE
-------------------
PMLI is assigned a shadow rating of BB (Moderate Risk), reflecting a 
weighted score of 2.75 out of 5.00. The rating is 
constrained primarily by elevated asset quality concerns and limited 
stress resilience, partially offset by adequate portfolio diversification 
and risk band distribution.

3. DIMENSION ANALYSIS
---------------------
a) Asset Quality [Score: 2/5 | Weight: 30%]
   Ba

In [11]:
# Save report as text file
report_path = '/content/drive/MyDrive/Credit_Risk_Framework/Outputs/shadow_rating_report.txt'

with open(report_path, 'w') as f:
    f.write(report)

print("Shadow rating report saved.")

Shadow rating report saved.
